In [2]:
import mlflow
import lightgbm
import pandas as pd
from dotenv import load_dotenv
import os
import sys
import json
import joblib

In [3]:
pd.set_option("display.max_columns", 100)
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
json_path = os.path.join(data_path, "processed/split_info.json")
with open(json_path) as f:
    json_info = json.load(f)
train_end = json_info.get("train_end")
val_end = json_info.get("validation_end")
from about_data.data_load import load_df
from about_data.split import temporal_split
from features.engineering import add_all_features
from model.preprocessor_pipe_evalueate import create_pipeline, evaluate_model, get_preprocessor

full_df = load_df(data_path)
train, val, test = temporal_split(full_df, train_end, val_end)

In [4]:
map_dfs = {"train": train, "val": val, "test": test}

x_dfs = {}
y_dfs = {}

for name, sample_df in map_dfs.items():
    x_dfs[name] = add_all_features(sample_df)
    y_dfs[name] = sample_df['isFraud']

In [11]:
model = lightgbm.LGBMClassifier(
    objective="binary",
    metric="average_precision",
    random_state=42,
    n_jobs=-1,
    
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=15,
    colsample_bytree=0.8,
    subsample=1.0,
    reg_alpha=0.0,
    reg_lambda=0.0,
    min_child_samples=20,
    
    is_unbalance=True 
)

pipe = create_pipeline(model, get_preprocessor(x_dfs['train']))

In [27]:
x_dfs['train'].shape[1]
X_train = pd.concat([x_dfs['train'], x_dfs['val']], axis=0)
y_train = pd.concat([y_dfs['train'], y_dfs['val']], axis=0)
X_test = x_dfs['test']
y_test = y_dfs['test']
X_train.shape

(501959, 44)

In [30]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("final_evaluation")
with mlflow.start_run(run_name="lightgbm_api_features"):
    pipe.fit(X_train, y_train)

    metrics = evaluate_model(pipe, X_test, y_test)

    mlflow.log_param("model", "lightgbm")

    mlflow.log_param("number_of_features", X_test.shape[1])

    mlflow.log_metrics(metrics)

[LightGBM] [Info] Number of positive: 17580, number of negative: 484379
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.104668 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15890
[LightGBM] [Info] Number of data points in the train set: 501959, number of used features: 6502
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035023 -> initscore=-3.316106
[LightGBM] [Info] Start training from score -3.316106
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [29]:
y_test

501959    0
501960    0
501961    0
501962    0
501963    0
         ..
590535    0
590536    0
590537    0
590538    0
590539    0
Name: isFraud, Length: 88581, dtype: int64